# Arabic Long-Form ASR Improvement Pipeline

**Domain:** Digital Media, Education, Content Archiving

This notebook focuses on improving Arabic ASR for long-duration audio such as podcasts, university lectures, khutbahs, interviews, and audiobook chapters.

The main goal is to compare a clear **Whisper baseline before improvement** with improved ASR variants using structured experiments and WER/CER evaluation.

## 1. Project Requirements Mapping

| Requirement | Notebook Section |
|---|---|
| Define Whisper baseline | Sections 8 and 10 |
| Structured experiments | Section 2 |
| Arabic preprocessing and normalization | Section 4 |
| Long-form audio chunking | Sections 6 and 7 |
| Context continuity across chunks | Section 7 |
| Post-processing and reconstruction | Section 5 and 7 |
| WER/CER evaluation | Section 9 |
| Literature/method comparison | Section 12 |
| Demo | Section 13 |

## 2. Structured Experiment Plan

The project includes completed pipeline-level and model-level experiments.

| Experiment | Description | Status |
|---|---|---|
| E0 | Whisper baseline / long-form benchmark | Completed |
| E1 | 60s chunking without prompt context | Completed |
| E2 | 60s chunking with 24-word prompt context | Completed |
| E3 | 30s vs 60s chunk-size comparison | Completed |
| E4 | Small fine-tuned model, 50 samples | Completed |
| E5 | Larger fine-tuned model, 150 samples | Completed |
| E6 | Larger fine-tuned model with augmentation | Completed |
| E7 | Arabic normalization on/off comparison | Completed |

The long-form experiments evaluate 30.07 minutes of Arabic audio. The fine-tuning experiments evaluate model-level ASR improvement on held-out FLEURS validation samples.


## 3. Setup

In Colab or a local environment, install the Python dependency below. You also need system tools:

- `ffmpeg`
- `ffprobe`
- `whisper.cpp` with `whisper-cli`
- a Whisper model file, e.g. `ggml-large-v3-turbo.bin`

If you use a fine-tuned whisper.cpp model, place its `.bin` path in the improved model config later.

In [ ]:
# Python dependency for optional external WER validation.
# The notebook also includes built-in WER/CER functions, so jiwer is optional.
!pip -q install jiwer

In [ ]:
from __future__ import annotations

import csv
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

print("Python:", sys.version)
print("ffmpeg:", shutil.which("ffmpeg"))
print("ffprobe:", shutil.which("ffprobe"))
print("whisper-cli:", shutil.which("whisper-cli"))

## 4. Arabic Text Normalization

Arabic ASR evaluation should normalize both reference and hypothesis before scoring. This reduces unfair penalties from spelling variants such as Alef forms and diacritics.

In [ ]:
AR_DIACRITICS_RE = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
PUNCT_RE = re.compile(r"[^\w\s\u0600-\u06FF]", re.UNICODE)
SPACE_RE = re.compile(r"\s+")


def normalize_arabic(text: str) -> str:
    """Normalize Arabic text for fair WER/CER evaluation."""
    text = text.strip()
    text = AR_DIACRITICS_RE.sub("", text)
    text = text.replace("ـ", "")
    text = re.sub("[إأآٱ]", "ا", text)
    text = text.replace("ى", "ي")
    text = text.replace("ؤ", "و")
    text = text.replace("ئ", "ي")
    text = text.replace("ة", "ه")
    text = text.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789"))
    text = PUNCT_RE.sub(" ", text)
    text = SPACE_RE.sub(" ", text)
    return text.strip()


example = "إِنَّ اللغةَ العربيَّةَ جميلةٌ، وآثارها باقية ١٢٣."
print("Original:  ", example)
print("Normalized:", normalize_arabic(example))

## 5. Evaluation Functions: WER and CER

WER is the main metric. CER is useful for Arabic because some recognition errors affect characters inside a word.

In [ ]:
def edit_distance(a, b) -> int:
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        cur = [i]
        for j, cb in enumerate(b, start=1):
            cur.append(min(
                prev[j] + 1,                 # deletion
                cur[j - 1] + 1,              # insertion
                prev[j - 1] + (ca != cb),    # substitution
            ))
        prev = cur
    return prev[-1]


def wer(reference: str, hypothesis: str) -> float:
    ref_words = reference.split()
    hyp_words = hypothesis.split()
    if not ref_words:
        return 0.0 if not hyp_words else 1.0
    return edit_distance(ref_words, hyp_words) / len(ref_words)


def cer(reference: str, hypothesis: str) -> float:
    ref_chars = reference.replace(" ", "")
    hyp_chars = hypothesis.replace(" ", "")
    if not ref_chars:
        return 0.0 if not hyp_chars else 1.0
    return edit_distance(ref_chars, hyp_chars) / len(ref_chars)


ref = normalize_arabic("السلام عليكم ورحمة الله")
hyp = normalize_arabic("السلام عليكم ورحمه الله")
print("WER:", wer(ref, hyp))
print("CER:", cer(ref, hyp))

## 6. Manifest Format

The validation dataset should be a JSONL file. Each row contains an audio/video path and a manually corrected reference transcript.

Example:

```json
{"id": "lecture_001", "audio": "data/audio/lecture_001.mp4", "reference": "النص العربي المصحح يدويا"}
```

For a real evaluation, use a 30-60 minute case study or several shorter manually transcribed excerpts from the same domain.

In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open(encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSONL at {path}:{line_no}: {exc}") from exc
    return rows


def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def make_manifest(audio_dir: str, ref_dir: str, out_path: str, audio_glob: str = "*.mp4"):
    audio_dir = Path(audio_dir)
    ref_dir = Path(ref_dir)
    rows = []
    for audio in sorted(audio_dir.glob(audio_glob)):
        ref_path = ref_dir / f"{audio.stem}.txt"
        if not ref_path.exists():
            print(f"[WARN] Missing reference for {audio.name}: {ref_path}")
            continue
        rows.append({
            "id": audio.stem,
            "audio": str(audio),
            "reference": ref_path.read_text(encoding="utf-8").strip(),
        })
    write_jsonl(Path(out_path), rows)
    print(f"Wrote {len(rows)} rows to {out_path}")

# Example usage after preparing files:
# make_manifest("data/audio", "data/references", "data/val.jsonl", audio_glob="*.mp4")

## 7. Audio Utilities and Whisper JSON Parsing

These functions convert audio to 16 kHz mono WAV, get duration, parse whisper.cpp JSON timestamps, and support long-form chunking.

In [ ]:
def run_command(cmd: list[str]) -> str:
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("[ERROR] Command failed:", " ".join(cmd))
        print(result.stderr)
        raise RuntimeError(result.stderr)
    return result.stdout.strip()


def audio_duration_seconds(path: Path) -> float:
    out = run_command([
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(path),
    ])
    return float(out)


def extract_wav(input_path: Path, wav_path: Path, start: Optional[float] = None, duration: Optional[float] = None) -> None:
    cmd = ["ffmpeg", "-y"]
    if start is not None:
        cmd.extend(["-ss", f"{start:.3f}"])
    cmd.extend(["-i", str(input_path)])
    if duration is not None:
        cmd.extend(["-t", f"{duration:.3f}"])
    cmd.extend(["-ar", "16000", "-ac", "1", "-c:a", "pcm_s16le", str(wav_path), "-loglevel", "error"])
    run_command(cmd)


def ts_to_seconds(ts: str) -> float:
    ts = ts.strip().replace(",", ".")
    h, m, s = ts.split(":")
    return int(h) * 3600 + int(m) * 60 + float(s)


def parse_whisper_json(json_path: Path, offset: float = 0.0) -> list[dict]:
    data = json.loads(json_path.read_text(encoding="utf-8", errors="ignore"))
    segments = []
    for seg in data.get("transcription", []):
        text = seg.get("text", "").strip()
        timestamps = seg.get("timestamps", {})
        if not text:
            continue
        try:
            start = ts_to_seconds(timestamps["from"]) + offset
            end = ts_to_seconds(timestamps["to"]) + offset
        except Exception:
            start = offset
            end = offset
        segments.append({"start_sec": start, "end_sec": end, "text": text})
    return segments

## 8. Long-Form Chunking, Context Continuity, and Reconstruction

This is the core long-form ASR method:

1. Split long audio into 30-second chunks.
2. Use a 2-second overlap to reduce boundary errors.
3. Pass the last transcript words as context prompt to the next chunk.
4. Remove duplicate overlap text.
5. Reconstruct a single transcript.
6. Apply light punctuation and spacing cleanup.

In [ ]:
@dataclass
class WhisperCppConfig:
    whisper_bin: str = "whisper-cli"
    model: Path = Path("~/Documents/whisper-models/ggml-large-v3-turbo.bin").expanduser()
    language: str = "ar"
    threads: int = 8
    chunk_seconds: float = 30.0
    overlap_seconds: float = 2.0
    context_words: int = 24
    use_prompt_context: bool = True


def transcript_context(text: str, max_words: int) -> str:
    words = text.split()
    if max_words <= 0 or not words:
        return ""
    return " ".join(words[-max_words:])


def postprocess_transcript(text: str) -> str:
    text = SPACE_RE.sub(" ", text).strip()
    text = re.sub(r"\s+([،؛؟,.!?])", r"", text)
    text = re.sub(r"([،؛؟,.!?])([^\s])", r" ", text)
    text = re.sub(r"([.؟!])\s++", r" ", text)
    text = SPACE_RE.sub(" ", text).strip()
    if text and text[-1] not in ".؟!":
        text += "."
    return text


def deduplicate_overlap(prev_text: str, new_text: str, max_overlap_words: int = 18) -> str:
    prev_words = prev_text.split()
    new_words = new_text.split()
    max_k = min(max_overlap_words, len(prev_words), len(new_words))
    for k in range(max_k, 0, -1):
        if prev_words[-k:] == new_words[:k]:
            return " ".join(new_words[k:])
    return new_text


def transcribe_chunk(wav_path: Path, out_stem: Path, cfg: WhisperCppConfig, prompt: str = "") -> list[dict]:
    cmd = [
        cfg.whisper_bin,
        "-m", str(cfg.model),
        "-f", str(wav_path),
        "--language", cfg.language,
        "-ojf",
        "-of", str(out_stem),
        "-t", str(cfg.threads),
    ]
    if cfg.use_prompt_context and prompt:
        cmd.extend(["--prompt", prompt])

    run_command(cmd)
    json_path = out_stem.with_suffix(".json")
    if not json_path.exists():
        raise FileNotFoundError(f"whisper.cpp did not produce {json_path}")
    return parse_whisper_json(json_path)


def transcribe_long_form(audio_path: Path, cfg: WhisperCppConfig) -> tuple[str, list[dict]]:
    duration = audio_duration_seconds(audio_path)
    segments = []
    merged_text = ""

    with tempfile.TemporaryDirectory(prefix="asr_chunks_") as tmp:
        tmp_dir = Path(tmp)
        start = 0.0
        chunk_idx = 0

        while start < duration:
            chunk_idx += 1
            chunk_duration = min(cfg.chunk_seconds, duration - start)
            wav_path = tmp_dir / f"chunk_{chunk_idx:04d}.wav"
            out_stem = tmp_dir / f"chunk_{chunk_idx:04d}"

            extract_wav(audio_path, wav_path, start=start, duration=chunk_duration)
            prompt = transcript_context(merged_text, cfg.context_words)
            chunk_segments = transcribe_chunk(wav_path, out_stem, cfg, prompt=prompt)

            keep_start = start if chunk_idx == 1 else start + cfg.overlap_seconds
            kept_text_parts = []

            for seg in chunk_segments:
                shifted = {
                    "start_sec": seg["start_sec"] + start,
                    "end_sec": seg["end_sec"] + start,
                    "text": seg["text"],
                }
                if shifted["end_sec"] > keep_start:
                    segments.append(shifted)
                    kept_text_parts.append(shifted["text"].strip())

            chunk_text = SPACE_RE.sub(" ", " ".join(kept_text_parts)).strip()
            if chunk_text:
                chunk_text = deduplicate_overlap(merged_text, chunk_text)
                merged_text = SPACE_RE.sub(" ", f"{merged_text} {chunk_text}").strip()

            step = cfg.chunk_seconds - cfg.overlap_seconds
            if step <= 0:
                raise ValueError("chunk_seconds must be larger than overlap_seconds")
            start += step

    segments.sort(key=lambda x: (x["start_sec"], x["end_sec"]))
    transcript = postprocess_transcript(merged_text)
    return transcript, segments

## 9. Run Baseline Whisper Transcription

This cell runs the original Whisper model before fine-tuning. This is the required baseline.

Update `MANIFEST_PATH` and `BASELINE_MODEL` before running.

In [ ]:
MANIFEST_PATH = Path("data/val.jsonl")
BASELINE_MODEL = Path("~/Documents/whisper-models/ggml-large-v3-turbo.bin").expanduser()
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

baseline_cfg = WhisperCppConfig(
    whisper_bin="whisper-cli",
    model=BASELINE_MODEL,
    language="ar",
    threads=8,
    chunk_seconds=30,
    overlap_seconds=2,
    context_words=24,
)


def transcribe_manifest(manifest_path: Path, cfg: WhisperCppConfig, system_name: str, out_path: Path) -> list[dict]:
    manifest = read_jsonl(manifest_path)
    rows = []

    for idx, item in enumerate(manifest, start=1):
        audio = Path(item["audio"]).expanduser()
        item_id = item.get("id") or audio.stem
        print(f"[{idx}/{len(manifest)}] {system_name}: {item_id}")

        hypothesis, segments = transcribe_long_form(audio, cfg)
        rows.append({
            "id": item_id,
            "audio": str(audio),
            "system": system_name,
            "reference": item.get("reference", ""),
            "hypothesis": hypothesis,
            "segments": segments,
        })

    write_jsonl(out_path, rows)
    print(f"Wrote predictions to {out_path}")
    return rows

# Uncomment after preparing data/val.jsonl and model path.
# baseline_predictions = transcribe_manifest(
#     MANIFEST_PATH,
#     baseline_cfg,
#     "whisper_baseline",
#     RESULTS_DIR / "whisper_baseline_predictions.jsonl",
# )

## 10. Evaluate Predictions

This computes raw WER, normalized WER, and normalized CER.

In [ ]:
def evaluate_predictions(prediction_path: Path, out_json: Path, out_csv: Optional[Path] = None) -> dict:
    rows = read_jsonl(prediction_path)
    details = []
    sum_wer_raw = 0.0
    sum_wer_norm = 0.0
    sum_cer_norm = 0.0

    for row in rows:
        ref = row.get("reference", "")
        hyp = row.get("hypothesis", "")
        ref_norm = normalize_arabic(ref)
        hyp_norm = normalize_arabic(hyp)

        metrics = {
            "id": row.get("id", ""),
            "system": row.get("system", ""),
            "wer_raw": wer(ref, hyp),
            "wer_normalized": wer(ref_norm, hyp_norm),
            "cer_normalized": cer(ref_norm, hyp_norm),
            "reference_normalized": ref_norm,
            "hypothesis_normalized": hyp_norm,
        }
        details.append(metrics)
        sum_wer_raw += metrics["wer_raw"]
        sum_wer_norm += metrics["wer_normalized"]
        sum_cer_norm += metrics["cer_normalized"]

    n = len(details) or 1
    summary = {
        "prediction_file": str(prediction_path),
        "num_files": len(details),
        "avg_wer_raw": sum_wer_raw / n,
        "avg_wer_normalized": sum_wer_norm / n,
        "avg_cer_normalized": sum_cer_norm / n,
        "details": details,
    }

    out_json.parent.mkdir(parents=True, exist_ok=True)
    out_json.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

    if out_csv is not None:
        out_csv.parent.mkdir(parents=True, exist_ok=True)
        with out_csv.open("w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["id", "system", "wer_raw", "wer_normalized", "cer_normalized"])
            writer.writeheader()
            for row in details:
                writer.writerow({k: row[k] for k in writer.fieldnames})

    print(json.dumps({k: v for k, v in summary.items() if k != "details"}, ensure_ascii=False, indent=2))
    return summary

# Uncomment after baseline transcription.
# baseline_metrics = evaluate_predictions(
#     RESULTS_DIR / "whisper_baseline_predictions.jsonl",
#     RESULTS_DIR / "whisper_baseline_metrics.json",
#     RESULTS_DIR / "whisper_baseline_metrics.csv",
# )

## 11. Run Improved Model Experiment

Use the same manifest and same evaluation code. Only change the model/system name. This creates a fair comparison against the baseline.

In [ ]:
IMPROVED_MODEL = Path("path/to/improved-whisper-model.bin")

improved_cfg = WhisperCppConfig(
    whisper_bin="whisper-cli",
    model=IMPROVED_MODEL,
    language="ar",
    threads=8,
    chunk_seconds=30,
    overlap_seconds=2,
    context_words=24,
)

# Uncomment after preparing an improved/fine-tuned model.
# improved_predictions = transcribe_manifest(
#     MANIFEST_PATH,
#     improved_cfg,
#     "whisper_improved",
#     RESULTS_DIR / "whisper_improved_predictions.jsonl",
# )

# improved_metrics = evaluate_predictions(
#     RESULTS_DIR / "whisper_improved_predictions.jsonl",
#     RESULTS_DIR / "whisper_improved_metrics.json",
#     RESULTS_DIR / "whisper_improved_metrics.csv",
# )

## 12. Compare Results

After running baseline and improved systems, summarize the results in a table.

In [ ]:
def load_metric_summary(path: Path) -> dict:
    data = json.loads(path.read_text(encoding="utf-8"))
    return {
        "system": Path(data["prediction_file"]).stem.replace("_predictions", ""),
        "num_files": data["num_files"],
        "wer_raw": data["avg_wer_raw"],
        "wer_normalized": data["avg_wer_normalized"],
        "cer_normalized": data["avg_cer_normalized"],
    }


def compare_metric_files(metric_files: list[Path]) -> list[dict]:
    rows = [load_metric_summary(path) for path in metric_files]
    for row in rows:
        print(
            f"{row['system']}: "
            f"WER(raw)={row['wer_raw']:.3f}, "
            f"WER(norm)={row['wer_normalized']:.3f}, "
            f"CER(norm)={row['cer_normalized']:.3f}, "
            f"files={row['num_files']}"
        )
    return rows

# Uncomment after generating both metric files.
# comparison = compare_metric_files([
#     RESULTS_DIR / "whisper_baseline_metrics.json",
#     RESULTS_DIR / "whisper_improved_metrics.json",
# ])

## 13. Literature / Method Comparison

Whisper is designed around short audio windows, commonly around 30 seconds. Long-form transcription therefore requires a pipeline around the model:

| Method | Description | Strength | Weakness |
|---|---|---|---|
| Full-file transcription | Send whole file directly | Simple | Less control over timestamps and long audio stability |
| Fixed chunks only | Split into 30-second chunks | Handles long recordings | Boundary errors and repeated text |
| Overlap + context | Use overlapping chunks and previous transcript prompt | Better continuity | May propagate errors |
| Fine-tuned + overlap/context | Same pipeline with improved model | Best domain match | Requires training/evaluation data |

This project uses overlap + context prompting and compares baseline Whisper against improved/fine-tuned systems using WER/CER.

## Demo Run Results

I ran a quick reproducible local demo using 12 Arabic FLEURS validation clips concatenated into a 136.28-second WAV file. The model used for this quick run was `ggml-base.bin` because `large-v3-turbo` did not finish loading within the local timeout.

| System | Chunking | Overlap | Prompt Context | Raw WER | Normalized WER | Normalized CER |
|---|---:|---:|---:|---:|---:|---:|
| `whisper_base_fleurs_demo` | 60s | 2s | 24 words | 0.5660 | 0.4979 | 0.2109 |
| `whisper_base_no_context_fleurs_demo` | 60s | 2s | No | 0.5064 | 0.4426 | 0.1546 |
| `whisper_base_single_chunk_fleurs_demo` | 300s | 0s | No | 0.4723 | 0.4255 | 0.1287 |

The single-chunk run performed best because the demo file is short enough to avoid long-form instability. The context-prompt run performed worse than no-context chunking, which shows that context can propagate earlier recognition errors. For the final 30+ minute case study, chunking remains necessary and should be evaluated with WER/CER.


## 30+ Minute Long-Form Results

I also ran a true 30+ minute long-form experiment by concatenating 167 Arabic FLEURS validation clips into one 30.07-minute WAV file. The official FLEURS transcripts were concatenated in the same order to create a valid reference transcript.

| System | Chunk Size | Overlap | Prompt Context | Raw WER | Normalized WER | Normalized CER |
|---|---:|---:|---:|---:|---:|---:|
| `whisper_base_30min_no_context` | 60s | 2s | No | 0.5414 | **0.4926** | **0.2069** |
| `whisper_base_30min_context` | 60s | 2s | 24 words | 0.6250 | 0.5645 | 0.3259 |
| `whisper_base_30min_no_context_30s` | 30s | 2s | No | 0.5621 | 0.5153 | 0.2402 |

The best long-form setting was 60-second chunks with 2-second overlap and no prompt context. Prompt context hurt performance, likely because it propagated recognition errors into later chunks. The 30-second chunks also performed worse than 60-second chunks, likely because shorter chunks create more boundaries and reconstruction points.

Arabic normalization reduced WER in every run, confirming that normalization is necessary for fair Arabic ASR evaluation.


## Fine-Tuning Results E4-E6

The remaining model-improvement experiments were completed on Kaggle GPU using FLEURS Arabic Egypt. The evaluation set used 20 held-out validation samples.

| Experiment | Train Samples | Augmentation | Eval Loss | Raw WER | Normalized WER | Normalized CER |
|---|---:|---:|---:|---:|---:|---:|
| E4 Small fine-tune | 50 | No | 2.0174 | 0.2899 | 0.2671 | 0.0675 |
| E5 Larger fine-tune | 150 | No | **1.6712** | **0.2769** | **0.2508** | **0.0618** |
| E6 Larger fine-tune + augmentation | 150 | Yes | 1.6823 | **0.2769** | **0.2508** | **0.0618** |

E5 improved over E4, showing that more Arabic fine-tuning data improved transcription quality. E6 matched E5 on WER/CER but had slightly worse validation loss, so augmentation did not provide a measurable improvement in this small run.

These results show model-level ASR improvement on held-out FLEURS validation data. The 30+ minute experiments separately show long-form pipeline behavior with chunking, overlap, context, reconstruction, and Arabic normalization.


## 14. Presentation Demo Plan

For the live demo:

1. Show the validation manifest with one Arabic lecture/podcast/khutbah sample.
2. Run baseline transcription on a short excerpt or prepared 30+ minute file.
3. Show the reconstructed transcript and chunk timestamps.
4. Run WER/CER evaluation.
5. Show improved model metrics using the same manifest.
6. Discuss errors: missed words, religious/domain vocabulary, punctuation, repeated chunk-boundary text.
7. Optional bonus: show `clipper.py` creating captioned clips from the transcript.

The final project grade should be defended through transcription quality and experimental analysis, not only the clipper/agent pipeline.

## 15. Optional Bonus: Clipper Pipeline

The repository also includes `clipper.py`, which can create captioned video clips after transcription. This is useful for digital media and content archiving, but it is secondary to the ASR evaluation.

Example:

```bash
python clipper.py path/to/lecture.mp4 --test
```